# 12-2절 연습 문제 풀이

이 노트북은 12-2절 연습 문제의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 구현도 얼마든지 가능하다.

- 본문 예제 코드는 `notebooks/ch12/` 아래 예제 노트북을 참고한다.
- 위에서부터 차례대로 실행한다.

In [ ]:
# 환경 설정 - 공통 라이브러리, 시드 고정, 장치 객체
import sys
sys.path.append('../../')

import random

import numpy as np
import torch
import torch.nn as nn

from code_reference import common
# viz.configure()에서 save_grayscale=True로 지정하면 노트북에 표시되는 시각화 이미지를 파일로 저장함
from code_reference import visualize as viz

viz.configure(save_grayscale=False)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = common.get_device()

# 12-2절 공통 - KoBART 뉴스 요약 미세 조정
# 주의: 모델과 데이터셋 내려받기, GPU가 필요하다.
try:
    from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM,
                              Seq2SeqTrainer, Seq2SeqTrainingArguments,
                              DataCollatorForSeq2Seq)
    from datasets import load_dataset
except ImportError:
    print('알림: pip install transformers datasets 로 라이브러리를 먼저 설치한다.')

MODEL_NAME = 'gogamza/kobart-base-v2'
DATASET_NAME = 'daekeun-ml/naver-news-summarization-ko'

## 연습 12-5

본문 예제에서 num_train_epochs를 2에서 5로 늘려 다시 학습한 뒤, [코드 12-11]의 같은 샘플로 출력 길이와 요약 품질이 어떻게 달라지는지 비교해 보자. 학습 로그에 기록되는 훈련 손실과 검증 손실의 변화도 함께 살펴보면 과적합이 시작되는 지점을 직접 확인할 수 있다.

In [ ]:
def finetune_kobart(epochs=2, train_size=500, out_dir='../../checkpoint/kobart-out'):
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
    raw = load_dataset(DATASET_NAME)
    train = raw['train'].select(range(train_size))
    valid = raw['validation'].select(range(min(100, len(raw['validation']))))

    def preprocess(batch):
        enc = tokenizer(batch['document'], max_length=512, truncation=True)
        enc['labels'] = tokenizer(text_target=batch['summary'],
                                  max_length=128, truncation=True)['input_ids']
        return enc

    train = train.map(preprocess, batched=True, remove_columns=train.column_names)
    valid = valid.map(preprocess, batched=True, remove_columns=valid.column_names)
    args = Seq2SeqTrainingArguments(output_dir=out_dir, num_train_epochs=epochs,
        per_device_train_batch_size=4, learning_rate=3e-5,
        eval_strategy='epoch', logging_steps=50, report_to=[])
    trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=train,
        eval_dataset=valid,
        data_collator=DataCollatorForSeq2Seq(tokenizer, model=model))
    trainer.train()
    return model, tokenizer

for epochs in (2, 5):
    print(f'=== num_train_epochs={epochs} ===')
    model, tokenizer = finetune_kobart(epochs=epochs)

에포크를 2에서 5로 늘리면 훈련 손실은 계속 줄지만 **검증 손실은 어느 시점부터 올라간다**(과적합). 요약 결과는 학습 데이터의 문체를 더 강하게 따라가고 출력이 짧아지는 경향이 있다.

요약 과제는 정답이 하나가 아니므로 손실만으로 판단하기 어렵다. 실제 출력을 함께 보고 판단해야 한다.

## 연습 12-6

TRAIN_SIZE를 500에서 2,000으로 늘려 다시 학습한 모델을 기존 모델과 비교해 학습 데이터셋 크기에 따른 학습 시간과 품질의 변화를 확인해 보자.

In [ ]:
import time
for size in (500, 2000):
    t0 = time.time()
    print(f'=== TRAIN_SIZE={size} ===')
    model, tokenizer = finetune_kobart(epochs=2, train_size=size,
                                       out_dir=f'../../checkpoint/kobart-{size}')
    print(f'학습 시간 {time.time() - t0:.1f}초')

데이터가 4배 늘면 **학습 시간도 대략 4배** 늘어난다. 품질은 대체로 좋아지지만 **수확 체감**이 뚜렷해서, 500 → 2,000의 개선 폭이 시간 증가만큼 크지는 않다.

미세 조정은 사전 학습 모델이 이미 언어를 알고 있기 때문에 적은 데이터로도 어느 정도 효과가 나온다는 점이 핵심이다.

## 연습 12-7

[도전 문제] 뉴스가 아닌 다른 분야의 한국어 텍스트로 원문과 요약문 쌍으로 이뤄진 작은 데이터셋을 만들고, 이를 사용해 KoBART 모델을 미세 조정해 보자.

In [ ]:
# 직접 만든 작은 데이터셋으로 미세 조정한다.
from datasets import Dataset
my_data = {
    'document': [
        '파이토치는 페이스북이 공개한 딥러닝 프레임워크로, 동적 계산 그래프를 사용해 '
        '디버깅이 쉽고 연구자들이 선호한다. 텐서 연산과 자동 미분을 기본으로 제공한다.',
        '합성곱 신경망은 이미지의 국소적인 패턴을 필터로 추출한다. 필터를 이미지 전체에 '
        '공유하므로 파라미터가 적고 위치 변화에 강하다.',
    ],
    'summary': ['파이토치는 동적 그래프 기반의 딥러닝 프레임워크다.',
                '합성곱 신경망은 필터 공유로 이미지 패턴을 효율적으로 추출한다.'],
}
ds = Dataset.from_dict(my_data)
print(ds)
print('\n실제 미세 조정은 위 finetune_kobart()의 데이터셋만 이 ds로 바꾸면 된다.')
print('데이터가 적을수록 학습률을 더 작게(1e-5 안팎) 두는 편이 안전하다.')

직접 만든 데이터셋은 **원문(document)과 요약(summary) 두 열**만 있으면 된다. 분야가 뉴스와 다를수록(예: 기술 문서, 논문 초록) 사전 학습 모델의 문체와 차이가 커서 더 많은 데이터가 필요하다.

데이터가 수십 건 수준이면 미세 조정보다 **프롬프트로 예시를 주는 방식**(퓨샷)이 나을 수도 있다.